# Generate and Save Quasistatic Trajectories

Pre-computes actuation trajectories for all trained networks in a specified
data directory and saves them alongside the network files as `trajectory_subtaskN.npz`.

Run this notebook **before** the figure notebooks to prepare the trajectory data.
Requires the Cython FIRE minimizer (built from `training/setup_fire_minimizer_memview_cython.py`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('../../../').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Project root:', REPO_ROOT)

## Parameters
Edit the cell below to control which networks are processed.

In [ ]:
# ── Network type ─────────────────────────────────────────────────────────────
NETWORK_TYPE = 'auxetic'   # 'auxetic' or 'allosteric'

# ── Data directories ─────────────────────────────────────────────────────────
# Defaults: change to point at your specific subfolder.
DATA_ROOT     = REPO_ROOT / 'data'

# For auxetic networks pick the subfolder:
SUBFOLDER     = 'targeted_results_sqr'   # e.g. 'results', 'targeted_results', 'targeted_results_sqr'
DATA_DIR      = DATA_ROOT / 'auxetic_nets' / SUBFOLDER

# For allosteric networks:
# DATA_DIR = DATA_ROOT / 'allosteric_nets'

# ── Trajectory parameters ────────────────────────────────────────────────────
N_STEPS       = None        # None → use config.get_n_strain_steps(task_seed) per task
FORCE_TYPE    = 'quadratic'
TOL           = 1e-6
OVERWRITE     = False       # Set True to recompute existing trajectories

# ── Task / realization filter ────────────────────────────────────────────────
# None = process all discovered networks; or specify lists to restrict.
TASK_FILTER   = None        # e.g. [0, 1, 2]
REAL_FILTER   = None        # e.g. [0]

print(f'DATA_DIR: {DATA_DIR}')
print(f'Exists:   {DATA_DIR.exists()}')

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

from analysis.data_io import load_auxetic_network
from analysis.trajectory import compute_auxetic_trajectory, save_trajectory
from base.config import get_n_strain_steps
from training.src.task_generator import generate_task_config

print('Imports OK')

## Discover networks

In [ ]:
pairs = []
for task_path in sorted(DATA_DIR.glob('task_*')):
    task_seed = int(task_path.name.split('_')[1])
    if TASK_FILTER is not None and task_seed not in TASK_FILTER:
        continue
    for real_path in sorted(task_path.glob('realization_*')):
        real_seed = int(real_path.name.split('_')[1])
        if REAL_FILTER is not None and real_seed not in REAL_FILTER:
            continue
        pkl = real_path / 'final_network.pkl'
        if pkl.exists():
            pairs.append((task_seed, real_seed))

print(f'Found {len(pairs)} networks in {DATA_DIR}')

## Compute and save trajectories

In [ ]:
errors = []

for task_seed, real_seed in tqdm(pairs, desc='Networks'):
    path = DATA_DIR / f'task_{task_seed:02d}' / f'realization_{real_seed:02d}'
    n_steps = N_STEPS if N_STEPS is not None else get_n_strain_steps(task_seed)
    task_cfg = generate_task_config(task_seed)
    compression_strains = task_cfg['compression_strains']

    try:
        network, boundary = load_auxetic_network(task_seed, real_seed, DATA_DIR)
    except Exception as e:
        errors.append((task_seed, real_seed, str(e)))
        print(f'  ERROR loading task={task_seed} real={real_seed}: {e}')
        continue

    for subtask_idx, cs in enumerate(compression_strains):
        out = path / f'trajectory_subtask{subtask_idx}.npz'
        if out.exists() and not OVERWRITE:
            continue
        try:
            traj = compute_auxetic_trajectory(
                network, cs, boundary, n_steps=n_steps,
                force_type=FORCE_TYPE, tol=TOL,
            )
            save_trajectory(traj, out)
        except Exception as e:
            errors.append((task_seed, real_seed, subtask_idx, str(e)))
            print(f'  ERROR task={task_seed} real={real_seed} subtask={subtask_idx}: {e}')

print(f'Done. Errors: {len(errors)}')
if errors:
    print(errors)